In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [2]:
import torch
import gc
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback, DataCollatorForSeq2Seq

c:\Users\ADMIN\anaconda3\envs\tracking-barbell-exercises\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
c:\Users\ADMIN\anaconda3\envs\tracking-barbell-exercises\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

# Kiểm tra lại version để chắc chắn
import accelerate
print(f"Check version: {accelerate.__version__}") 

from transformers import (
    AutoModelForSeq2SeqLM, 
    AutoTokenizer, 
    DataCollatorForSeq2Seq, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer,
    EarlyStoppingCallback
)
from datasets import Dataset

Check version: 1.0.1


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang dùng: {device}")

Đang dùng: cuda


In [6]:
import pandas as pd

In [ ]:
import pandas as pd
try:
    # Sử dụng dấu gạch chéo hoặc tiền tố r'' để tránh lỗi escape
    data = pd.read_csv(r'C:\Code\TeenCodeTranslator\process_thread_data\train_data.csv', encoding='utf-8')
    print('Đã load thành công train_data.csv!')
    print(data.head())
except Exception as e:
    print('Lỗi khi load file train_data.csv:', e)

Lỗi khi load file train_data.csv: [Errno 22] Invalid argument: 'C:\\Code\\TeenCodeTranslator\\process_thread_data\train_data.csv'


In [ ]:
# --- KHỞI ĐỘNG TENSORBOARD LIVE ---
%load_ext tensorboard
%tensorboard --logdir ./logs

# --- PHẦN FINE-TUNE ---
model_name = "vinai/bartpho-syllable"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Giả sử 'data' đã được chuẩn hóa ở các bước trước
raw_dataset = Dataset.from_pandas(data[['text', 'target']])

def preprocess_function(examples):
    model_inputs = tokenizer(examples["sentence"], max_length=64, truncation=True, padding="max_length")
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples["output"], max_length=64, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = raw_dataset.map(preprocess_function, batched=True)
split_dataset = tokenized_dataset.train_test_split(test_size=0.1)

learning_rates = [2e-5, 5e-5, 6e-5, 8e-5, 1e-4]

for lr in learning_rates:
    run_name = f"Trial_LR_{lr}"
    print(f"\n>>> ĐANG CHẠY TRIAL: {run_name}")
    
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    
    training_args = Seq2SeqTrainingArguments(
            output_dir=f"./outputs/{run_name}",
            evaluation_strategy="steps",
            eval_steps=200,                # Đánh giá sau mỗi 200 bước
            learning_rate=lr,              # Nên dùng list: [1e-5, 2e-5, 3e-5]
            
            # --- Tối ưu Batch Size ---
            per_device_train_batch_size=16, 
            per_device_eval_batch_size=16,
            gradient_accumulation_steps=4, # Effective batch size = 16 * 4 = 64
            
            # --- Tối ưu Huấn luyện ---
            num_train_epochs=10,           # Cho phép học lâu hơn
            warmup_ratio=0.1,              # Khởi động mềm mỏng
            lr_scheduler_type="cosine",    # Giảm tốc độ học mượt mà
            weight_decay=0.01,             # Chống học vẹt
            label_smoothing_factor=0.1,    # Tăng độ linh hoạt khi sinh từ
            
            # --- Hiệu suất & Lưu trữ ---
            fp16=True,                      
            logging_dir=f"./logs/{run_name}",
            logging_steps=50,               
            report_to="tensorboard",        
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,        # Loss càng thấp càng tốt
            gradient_checkpointing=True,    
            save_total_limit=2              # Lưu 2 bản checkpoint gần nhất để an toàn
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=split_dataset["train"],
        eval_dataset=split_dataset["test"],
        tokenizer=tokenizer,
        data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )

    trainer.train()